In [34]:
!pip install langchain-openai python-docx -q

In [35]:
from datetime import datetime
from pathlib import Path
import os
import re
import ast
import math
import json
import urllib.parse
import urllib.request
import urllib.error
import operator as op
import docx

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="not-needed",
    model="google/gemma-3-4b",
    temperature=0.2,
    timeout=45,
    max_retries=1
)

print("✓ Imports loaded successfully")
print("✓ Connected to Gemma 3 4B")
print("  API: http://127.0.0.1:1234/v1")
print("  Model: google/gemma-3-4b")
print("  Temperature: 0.2")
print("  Timeout: 45 seconds")


✓ Imports loaded successfully
✓ Connected to Gemma 3 4B
  API: http://127.0.0.1:1234/v1
  Model: google/gemma-3-4b
  Temperature: 0.2
  Timeout: 45 seconds


In [36]:
OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
}

ALLOWED_FUNCTIONS = {
    "sqrt": math.sqrt,
    "abs": abs,
    "round": round,
}

ALLOWED_NAMES = {
    "pi": math.pi,
    "e": math.e,
}

def safe_eval(expr: str):
    def _eval(node):
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return node.value
            raise ValueError("Only numeric values are allowed")

        elif isinstance(node, ast.Num):  # compatibility for older Python
            return node.n

        elif isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in OPS:
                raise ValueError("Operator not allowed")
            return OPS[op_type](_eval(node.left), _eval(node.right))

        elif isinstance(node, ast.UnaryOp):
            if isinstance(node.op, ast.USub):
                return -_eval(node.operand)
            elif isinstance(node.op, ast.UAdd):
                return +_eval(node.operand)
            raise ValueError("Unary operator not allowed")

        elif isinstance(node, ast.Call):
            if not isinstance(node.func, ast.Name):
                raise ValueError("Invalid function call")

            func_name = node.func.id
            if func_name not in ALLOWED_FUNCTIONS:
                raise ValueError(f"Function not allowed: {func_name}")

            args = [_eval(arg) for arg in node.args]
            return ALLOWED_FUNCTIONS[func_name](*args)

        elif isinstance(node, ast.Name):
            if node.id in ALLOWED_NAMES:
                return ALLOWED_NAMES[node.id]
            raise ValueError(f"Unknown constant: {node.id}")

        else:
            raise ValueError("Invalid expression")

    tree = ast.parse(expr, mode="eval")
    return _eval(tree.body)

def tool_calculator(query: str) -> str:
    try:
        expr = query.strip()
        result = safe_eval(expr)
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"

print("✓ Calculator tool ready")

✓ Calculator tool ready


In [37]:
def fetch_json(url: str, timeout: int = 10) -> dict:
    headers = {
        "User-Agent": "GemmaAssistant/1.0 (Educational Project)"
    }
    req = urllib.request.Request(url, headers=headers)

    with urllib.request.urlopen(req, timeout=timeout) as response:
        raw = response.read().decode("utf-8")
        return json.loads(raw)

def tool_wikipedia(query: str) -> str:
    try:
        query = query.strip()
        if not query:
            return "Wikipedia error: empty query."

        # Step 1: search for the best matching title
        search_url = (
            "https://en.wikipedia.org/w/api.php?"
            + urllib.parse.urlencode({
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "utf8": 1,
                "srlimit": 1
            })
        )

        search_data = fetch_json(search_url)
        search_results = search_data.get("query", {}).get("search", [])

        if not search_results:
            return f"No Wikipedia results found for '{query}'."

        best_title = search_results[0]["title"]

        # Step 2: fetch plain-text extract
        extract_url = (
            "https://en.wikipedia.org/w/api.php?"
            + urllib.parse.urlencode({
                "action": "query",
                "prop": "extracts",
                "exintro": 1,
                "explaintext": 1,
                "titles": best_title,
                "format": "json",
                "utf8": 1
            })
        )

        extract_data = fetch_json(extract_url)
        pages = extract_data.get("query", {}).get("pages", {})

        if not pages:
            return f"Found '{best_title}', but no page content was available."

        page = next(iter(pages.values()))
        extract = page.get("extract", "").strip()

        if extract:
            return extract

        return f"Found '{best_title}', but no summary was available."

    except json.JSONDecodeError:
        return "Wikipedia error: invalid JSON response."
    except urllib.error.HTTPError as e:
        return f"Wikipedia HTTP error: {e.code} {e.reason}"
    except urllib.error.URLError as e:
        return f"Wikipedia network error: {e.reason}"
    except Exception as e:
        return f"Wikipedia error: {e}"

print("✓ Wikipedia tool ready")

✓ Wikipedia tool ready


In [38]:
def sanitize_filename(text: str, max_len: int = 80) -> str:
    text = re.sub(r"[^\w\s-]", "", text).strip().replace(" ", "_")
    text = re.sub(r"_+", "_", text)
    return text[:max_len] if text else "generated_document"

def extract_document_topic(query: str) -> str:
    q = query.strip()

    patterns = [
        r"(?i)create\s+(?:a\s+)?document\s+about\s+(.+)",
        r"(?i)generate\s+(?:a\s+)?document\s+about\s+(.+)",
        r"(?i)write\s+(?:a\s+)?document\s+about\s+(.+)",
        r"(?i)make\s+(?:a\s+)?document\s+about\s+(.+)",
        r"(?i)create\s+(?:a\s+)?word\s+document\s+about\s+(.+)",
        r"(?i)generate\s+(?:a\s+)?word\s+document\s+about\s+(.+)",
        r"(?i)write\s+(?:a\s+)?report\s+on\s+(.+)",
        r"(?i)create\s+(?:a\s+)?report\s+on\s+(.+)",
        r"(?i)generate\s+(?:a\s+)?report\s+on\s+(.+)",
        r"(?i)make\s+(?:a\s+)?file\s+about\s+(.+)",
        r"(?i)create\s+(?:a\s+)?file\s+about\s+(.+)",
        r"(?i)generate\s+(?:a\s+)?file\s+about\s+(.+)",
        r"(?i)document\s+about\s+(.+)",
        r"(?i)report\s+about\s+(.+)",
        r"(?i)about\s+(.+)",
        r"(?i)on\s+(.+)",
    ]

    for pattern in patterns:
        m = re.match(pattern, q)
        if m:
            topic = m.group(1).strip()
            topic = topic.rstrip(".!?")
            return topic

    return q.rstrip(".!?")

def format_topic_title(topic: str) -> str:
    return topic.strip().title() if topic.strip() else "Generated Document"

def create_word_doc(content: str, title: str, topic: str, folder: str = "generated_docs") -> str:
    os.makedirs(folder, exist_ok=True)

    safe_topic = sanitize_filename(topic)
    filename = f"{safe_topic}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
    filepath = os.path.join(folder, filename)

    doc = docx.Document()
    doc.add_heading(title, 0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph("")

    for line in content.split("\n"):
        line = line.strip()
        if not line:
            continue

        if line.startswith("## "):
            doc.add_heading(line[3:], level=1)
        elif line.startswith("### "):
            doc.add_heading(line[4:], level=2)
        elif line.startswith("- "):
            doc.add_paragraph(line[2:], style="List Bullet")
        else:
            doc.add_paragraph(line)

    doc.save(filepath)
    return filepath

def basic_document_template(topic: str) -> str:
    return f"""## Overview
This document is about {topic}.

## Main Points
- Introduction to the topic
- Important background information
- Key ideas and concepts
- Practical relevance or examples

## Summary
This is a fallback document generated without waiting for the language model response.
You can improve or regenerate it later when the model server is available.
"""

def word_tool(query: str) -> str:
    try:
        query = query.strip()
        if not query:
            return "Document error: empty request."

        topic = extract_document_topic(query)
        title = format_topic_title(topic)

        prompt = f"""
Create a professional, well-structured document about this topic:

{topic}

Requirements:
- Start with a clear title
- Use clear sections
- Use bullet points where helpful
- Keep it informative and readable
- Return plain text only
- Use headings like:
  ## Section
  ### Subsection
"""

        try:
            content = llm.invoke(prompt).content.strip()
            if not content:
                content = basic_document_template(topic)
        except Exception:
            content = basic_document_template(topic)

        path = create_word_doc(content, title=title, topic=topic)
        return f"Document created successfully: {path}"

    except Exception as e:
        return f"Error creating document: {e}"

print("✓ Document generator tool ready")
print("  • Filename uses the topic")
print("  • Document title uses the topic")

✓ Document generator tool ready
  • Filename uses the topic
  • Document title uses the topic


In [39]:
DOCUMENT_KEYWORDS = [
    "create document",
    "generate document",
    "write document",
    "make document",
    "create a doc",
    "generate word",
    "word document",
    "make a file",
    "create a file",
    "generate a file",
    "write a report",
    "create report",
    "generate report",
    "make a report",
    "document about",
    "report about",
]

CALCULATOR_KEYWORDS = [
    "calculate",
    "calculation",
    "compute",
    "solve",
    "evaluate",
    "how much is",
    "sum",
    "difference",
    "product",
    "quotient",
    "add",
    "subtract",
    "multiply",
    "divide",
    "plus",
    "minus",
    "times",
    "multiplied by",
    "divided by",
    "mod",
    "remainder",
    "power",
    "to the power of",
    "square root",
    "sqrt",
    "percent",
    "percentage",
]

def is_document_request(query: str) -> bool:
    q = query.lower().strip()
    return any(keyword in q for keyword in DOCUMENT_KEYWORDS)

def is_calculation_query(query: str) -> bool:
    q = query.lower().strip()

    if re.fullmatch(r"[\d\s\.\+\-\*\/\(\)\%\^]+", q):
        return True

    if re.search(r"\d", q) and re.search(r"[\+\-\*\/\%\^]", q):
        return True

    if any(keyword in q for keyword in CALCULATOR_KEYWORDS) and re.search(r"\d", q):
        return True

    if "square root" in q or "sqrt" in q:
        return True

    if re.search(r"\bpi\b", q):
        return True

    if re.search(r"\bvalue of e\b", q):
        return True

    patterns = [
        r"add\s+\d+(\.\d+)?\s+(and|to)\s+\d+(\.\d+)?",
        r"subtract\s+\d+(\.\d+)?\s+from\s+\d+(\.\d+)?",
        r"multiply\s+\d+(\.\d+)?\s+by\s+\d+(\.\d+)?",
        r"divide\s+\d+(\.\d+)?\s+by\s+\d+(\.\d+)?",
        r"\d+(\.\d+)?\s+(percent|percentage)\s+of\s+\d+(\.\d+)?",
    ]
    if any(re.search(pattern, q) for pattern in patterns):
        return True

    return False

def extract_math_expression(query: str) -> str:
    q = query.lower().strip()

    removable_phrases = [
        "what is",
        "how much is",
        "calculate",
        "compute",
        "solve",
        "evaluate",
        "please",
        "the value of",
        "value of",
    ]
    for phrase in removable_phrases:
        q = q.replace(phrase, "")

    q = q.strip()

    q = re.sub(r"square root of\s+(\d+(\.\d+)?)", r"sqrt(\1)", q)
    q = re.sub(r"sqrt\s+(\d+(\.\d+)?)", r"sqrt(\1)", q)

    q = re.sub(
        r"(\d+(\.\d+)?)\s*(percent|percentage)\s+of\s+(\d+(\.\d+)?)",
        r"(\1/100)*\4",
        q
    )

    m = re.match(r"add\s+(\d+(\.\d+)?)\s+(and|to)\s+(\d+(\.\d+)?)", q)
    if m:
        return f"{m.group(1)} + {m.group(4)}"

    m = re.match(r"subtract\s+(\d+(\.\d+)?)\s+from\s+(\d+(\.\d+)?)", q)
    if m:
        return f"{m.group(3)} - {m.group(1)}"

    m = re.match(r"multiply\s+(\d+(\.\d+)?)\s+by\s+(\d+(\.\d+)?)", q)
    if m:
        return f"{m.group(1)} * {m.group(3)}"

    m = re.match(r"divide\s+(\d+(\.\d+)?)\s+by\s+(\d+(\.\d+)?)", q)
    if m:
        return f"{m.group(1)} / {m.group(3)}"

    replacements = {
        "multiplied by": "*",
        "times": "*",
        "divided by": "/",
        "over": "/",
        "plus": "+",
        "minus": "-",
        "mod": "%",
        "remainder": "%",
        "to the power of": "**",
        "power of": "**",
        "^": "**",
    }

    for old, new in replacements.items():
        q = q.replace(old, new)

    q = re.sub(r"(?<=\d)\s*x\s*(?=\d)", " * ", q)

    q = q.replace("?", "").replace(",", " ")
    q = re.sub(r"\s+", " ", q).strip()

    return q

def route_query(query: str) -> str:
    if is_document_request(query):
        result = word_tool(query)
        return f"Tool used: Document Generator\n{result}"

    if is_calculation_query(query):
        expr = extract_math_expression(query)
        result = tool_calculator(expr)
        return f"Tool used: Calculator\nExpression: {expr}\nResult: {result}"

    result = tool_wikipedia(query)
    return f"Tool used: Wikipedia Search\n{result}"

def handle_query(query: str) -> str:
    return route_query(query)

print("✓ Router ready")
print("  • Document requests -> Word generator")
print("  • Calculation requests -> Calculator (forced)")
print("  • Other information requests -> Wikipedia")

✓ Router ready
  • Document requests -> Word generator
  • Calculation requests -> Calculator (forced)
  • Other information requests -> Wikipedia


In [40]:
print("\n" + "=" * 80)
print("GEMMA AI ASSISTANT - 3 TOOL VERSION")
print("=" * 80)
print("\nAvailable Tools:")
print("  • Calculator")
print("  • Wikipedia Search")
print("  • Document Generator")
print("\nType 'exit' or 'quit' to stop.")
print("=" * 80 + "\n")

while True:
    try:
        query = input("You: ").strip()

        if not query:
            continue

        if query.lower() in ["exit", "quit", "q"]:
            print("\n✓ Chat ended. Goodbye!\n")
            break

        print("\nGEMMA: Processing...\n")
        response = handle_query(query)

        print("=" * 80)
        print(f"QUESTION: {query}")
        print("-" * 80)
        print(f"GEMMA: {response}")
        print("=" * 80 + "\n")

    except KeyboardInterrupt:
        print("\n✓ Chat interrupted. Goodbye!\n")
        break
    except Exception as e:
        print("\n" + "=" * 80)
        print(f"QUESTION: {query}")
        print("-" * 80)
        print(f"GEMMA: ❌ System error: {e}")
        print("=" * 80 + "\n")



GEMMA AI ASSISTANT - 3 TOOL VERSION

Available Tools:
  • Calculator
  • Wikipedia Search
  • Document Generator

Type 'exit' or 'quit' to stop.


GEMMA: Processing...

QUESTION: 10 TIMES 25
--------------------------------------------------------------------------------
GEMMA: Tool used: Calculator
Expression: 10 * 25
Result: 250



C:\Users\jeric\AppData\Local\Temp\ipykernel_23196\630220777.py:28: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif isinstance(node, ast.Num):  # compatibility for older Python



GEMMA: Processing...

QUESTION: WHAT IS RECYCLING?
--------------------------------------------------------------------------------
GEMMA: Tool used: Wikipedia Search
Recycling is the process of converting waste materials into new materials and objects. This concept often includes the recovery of energy from waste materials. The recyclability of a material depends on its ability to reacquire the properties it had in its original state. It is an alternative to "conventional" waste disposal that can save material and help lower greenhouse gas emissions. It can also prevent the waste of potentially useful materials and reduce the consumption of fresh raw materials, reducing energy use, air pollution (from incineration) and water pollution (from landfilling).
Recycling is a key component of modern waste reduction and represents the third step in the "Reduce, Reuse, and Recycle" waste hierarchy, contributing to environmental sustainability and resource conservation. It promotes environment

In [41]:
print("\n" + "=" * 80)
print("GENERATED DOCUMENTS SUMMARY")
print("=" * 80 + "\n")

doc_dir = Path("generated_docs")

if doc_dir.exists():
    files = sorted(list(doc_dir.glob("*.docx")), key=lambda x: x.stat().st_mtime, reverse=True)

    if files:
        print(f"✓ Total Documents Created: {len(files)}\n")
        print(f"{'#':<4} {'Filename':<55} {'Size':<12} {'Modified':<20}")
        print("-" * 100)

        for i, f in enumerate(files, 1):
            size_kb = f.stat().st_size / 1024
            mod_time = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
            filename = f.name[:52] + "..." if len(f.name) > 52 else f.name
            print(f"{i:<4} {filename:<55} {size_kb:>8.1f} KB   {mod_time:<20}")
    else:
        print("✓ No documents generated yet.")
else:
    print("✓ Documents folder not created yet.")


GENERATED DOCUMENTS SUMMARY

✓ Total Documents Created: 1

#    Filename                                                Size         Modified            
----------------------------------------------------------------------------------------------------
1    SOLAR_SYSTEM_20260429_013858.docx                           37.7 KB   2026-04-29 01:38:58 
